# 自改进智能体的开放进化

> 前六讲回答的是同一个问题：人怎么设计一个更好的 **Agent**。第 4 讲搭出 ReAct 循环，第 5 讲设计任务分解与树搜索，第 6 讲用**强化学习**训练 Agent 的参数。设计者始终是人，Agent 只是被设计的对象。
>
> 这一讲把设计者本身交给搜索：让一个 Agent 去设计另一个 **Agent**——生成候选代码、在任务上打分、把好的结果存进 archive、再从这些结果里得到启发生成下一批。这套变异、选择、积累的循环就是**开放进化**。我们从 ADAS 的最小闭环出发，放大到 AI Scientist 的科研流程，再看 AlphaEvolve 把整份代码当基因组来进化，最后讨论它的失败模式。

先看一个最小的例子。一个只会“直接算”两位数乘法的 Agent，比如算 23×47，单次准确率只有 0.55——一大半题算错。

让另一个程序读它的代码、动手改。第一处改动：把“直接算”换成“先拆开再算”（先算 23×40、再算 23×7、再相加）。重新评测，准确率从 0.55 跳到 0.80。

第二处改动：在拆开算之后再加一步“验算”（换一种顺序重算一遍，对不上就重来）。准确率再升到 0.93。

就这么改一改、跑一跑、把分更高的留下来，几轮下来 Agent 自己越来越强。这种“让程序去改 Agent 的代码、越改越好”的循环，就是这一讲的主题——**开放进化**。

它和第 6 讲的训练不一样。训练改的是模型参数，训练一结束能力就定了；开放进化改的是更上层的东西——Agent 的代码、研究的想法、算法的结构。这一讲用三个真实系统来讲：ADAS 进化 Agent 的代码、AI Scientist 进化研究想法、AlphaEvolve 进化整份源码。

这套循环也有风险。系统自己给自己的产出打分时，一旦分数和真实目标脱节，它就专找取巧的路——分虚高、事没做成。这种失败叫**奖励黑客**，评测的独立性是唯一的护栏。

下面先看最基本的一步：一个程序怎样去改动另一个 Agent 的代码。

开放进化里，负责改动 Agent 的是一个程序，叫**元 Agent**。这一节搭一套最小的系统：元 Agent 读已有的 Agent 代码，写出新代码，在任务上评测，把好的留下来。这套系统叫 **ADAS**（Automated Design of Agentic Systems），它把“设计 Agent”当成一个搜索问题来解。

先约定 Agent 长什么样。ADAS 把一个 Agent 的全部组件——提示词、工具调用、控制流——都塞进同一个函数 `forward(task)` 里。函数怎么写，Agent 就怎么行动；所以“搜索更好的 Agent”就是“搜索 forward 的写法”。

代码的写法太多了，人没法一个个试。ADAS 靠一个循环来搜，循环转起来靠三样东西：**变异**产生新代码（由大模型扮演，读旧代码、写新代码）、**选择**判断好不好（一段评测函数，在固定任务上跑、返回准确率）、**种群**把好的留下来（一个叫 archive 的字典，存名字、代码、得分，供下一轮参考）。

这一节先做两件准备：把任务和评测函数定下来。

设计 Agent 有两个层次。低层次是调参数：换提示词里的词、改推理步数、改采样次数。高层次是改结构：换一种决策方式，甚至换一套控制流。ADAS 搜的是高层次——它的搜索空间是 Agent 的源代码，不是一组参数。

为什么代码能当搜索空间。Python 的表达力够强：条件、循环、递归、调用别的函数，任何能算的逻辑都能写出来。这种“什么都能表达”的性质有个名字，叫**图灵完备**。空间大到人没想过的方案也可能藏在里面，代价是太大，必须靠循环来搜。

用一个最小的例子看代码怎么承载策略。基线 Agent 的 forward 只有一行：

```text
def forward(task):
    a, b = task
    return solve(a, b, recipe='direct')
```

recipe 取什么值，Agent 就用什么策略：取 direct 是直接算，取 decompose 是先拆开再算，取 ensemble 是同时跑几个配方再投票。换一个词，行为就换一种。真实系统里 forward 还能写 if、for、多次调用，控制流越丰富，能表达的策略越多。

要让这个空间能搜，还得有两样东西：一份固定的任务，和一把给方案打分的标尺（评测函数）。下面把它们定下来。

In [ ]:
# 任务：两位数乘法。框架提供 solve(a, b, recipe)，按"推理配方"计算答案；
# 环境用确定性伪随机判断每个配方会在哪些题上失误，模拟底层模型
# 在不同推理深度下的差错率。np.random.seed 保证后续演示可复现。
import numpy as np
np.random.seed(42)

A = np.repeat(np.arange(11, 16), 8)   # 5 个十位，每个重复 8 次
B = np.tile(np.arange(11, 19), 5)     # 8 个个位，平铺 5 份
PROBLEMS = list(zip(A.tolist(), B.tolist()))
TRUE = [a * b for a, b in PROBLEMS]

RECIPE_RATE = {"direct": 0.40, "decompose": 0.12,
               "decompose_check": 0.05, "ensemble": 0.00}

def is_tricky(a, b, recipe):
    """该配方在这个题上是否失误：由种子决定的确定性判断。"""
    r = np.random.RandomState(1000 * (a % 10) + (b % 10) + len(recipe))
    return r.rand() < RECIPE_RATE[recipe]

def solve(a, b, recipe):
    """基础求解函数：按配方计算 a*b，失误时十位与个位交换。"""
    if is_tricky(a, b, recipe):
        return str((a % 10) * (b % 10) + 100 * (a // 10) * (b // 10))
    return str(a * b)

for rec in RECIPE_RATE:
    acc = sum(1 for (a, b), t in zip(PROBLEMS, TRUE)
              if solve(a, b, rec) == str(t)) / len(PROBLEMS)
    print(f"配方 {rec:<14} 设定失误率 {RECIPE_RATE[rec]:.2f} 实测准确率 {acc:.2f}")

上面这段代码跑出了四种配方的准确率，正好告诉我们每个方案做得怎么样。用这个数字判断方案好坏，就是选择环节要用的分数，这个分数有一个专门的名字，叫适应度。把结果整理成一张表：

| 配方 | 设定失误率 | 40 道题实测准确率 |
|:---|:---|:---|
| direct | 0.40 | 0.55 |
| decompose | 0.12 | 0.80 |
| decompose_check | 0.05 | 0.93 |
| ensemble | 0.00 | 1.00 |

失误率越高，准确率越低；失误率 0 的 ensemble 一道题都没错。现在把开放进化的三个组件在这份代码里认出来。

变异算子负责产生新变体，它读已有的 Agent 代码，写出一份新的 forward，本节里由大模型扮演。选择机制负责判断哪个变体更好，它是一段评测函数，在固定任务上运行变体，返回准确率，这个准确率就是适应度。种群负责存放变体与得分，让新变体不是凭空出现，而是有前人的结果可以参考，本节里种群就是 archive 字典。

三个组件缺一不可。没有变异，我们只有已有的 Agent，循环无法前进；没有选择，好坏没有区分，搜索没有方向；没有 archive，每一轮从零开始，无法积累。变异负责探索，选择负责判断，archive 负责记忆，三者凑在一起才形成一个能自我改进的循环。

先看变异算子最小的样子：改一个词，评分就变。

这一小节回答一个问题：变异算子到底怎么产生新变体。在代码空间里，变异是改一行代码；在参数空间里，变异是改一个参数值。我们先看参数层面的变异：把 Agent 的提示词换一个词，评分跟着变。真实系统里这个评分来自实际运行 Agent，这里先用一个人造的确定性评分函数代替，方便观察变异的效果。这个人造函数里，每个动作关键词都带来固定的准确率增益。

In [ ]:
# 变异算子最小演示：提示词里插入一个动作关键词，评分改变。
# 评分函数是确定性的：基础准确率 0.30，叠加命中关键词的增益。
KEYWORD_BONUS = {"分解": 0.15, "验算": 0.15, "估算": 0.10, "分步": 0.05}

def prompt_score(prompt):
    """提示词评分：基础准确率 0.30，叠加命中关键词的增益。"""
    score = 0.30
    for word, bonus in KEYWORD_BONUS.items():
        if word in prompt:
            score += bonus
    return score

CANDIDATES = ["分解", "验算", "估算", "分步"]

def mutate_prompt(prompt, seed):
    """从候选词里随机挑一个追加到提示词末尾，完成一次变异。"""
    rng = np.random.RandomState(seed)
    word = CANDIDATES[int(rng.randint(len(CANDIDATES)))]
    return prompt + "，" + word

base = "请计算这道乘法题"
for k in range(4):
    child = mutate_prompt(base, k)
    print(f"变异 {k + 1}: 「{child}」 评分 {prompt_score(child):.2f}")

print("原始提示词评分:", round(prompt_score(base), 2))

运行结果里，原始提示词评分 0.30，四轮变异后分别是 0.45、0.45、0.45、0.40。每个候选词都有固定的加分：分解 +0.15、验算 +0.15、估算 +0.10、分步 +0.05。评分函数是确定性的，同样的提示词永远得到同样的分数，这样我们才能干净地观察变异到评分的因果关系。

这里的评分函数是人造的，故意设计成命中关键词就加分。真实系统里评分来自实际运行：把这份提示词交给 Agent，在 40 道题上跑一遍，用准确率当分数。人造评分的价值是让第一次接触变异的读者只看到机制本身，改一个词分数变多少，看得清清楚楚。

改一个词评分就变这件事很重要，因为进化靠分数差异来导航。如果任何改动都不影响分数，变异就失去意义，选择也无从下手；如果改动能让分数上下变化，变异算子就有了信号，得分变高的变异值得保留，得分变低的可以丢弃。这一轮我们只观察，不做选择。评分函数把变异到分数的关系固定下来，接下来的参数空间搜索把选择接进循环。

这一小节把选择接进循环，回答一个问题：给定一个评分函数，搜索算法怎么找到高分的参数。变异产生新变体，选择交给评分函数：分数高的变体被保留，分数低的被丢弃。我们把 Agent 的三个参数——推理步数、集成样本数、工具数——当成一个三维点 $p=(n_{cot}, n_{samples}, n_{tools})$，评分函数对每个点返回一个确定性的分数。

$$score(p) = 0.20 + 0.30·tanh(n_{cot}/3) + 0.10·tanh(n_{samples}/4) + 0.05·tanh((n_{tools}-2)/1.5) - 0.03·max(0, n_{tools}-5)$$

这个公式算的是"给定一组参数，Agent 大概能做到多好"。前三项让推理步数与集成样本数产生边际递减的提升，最后一项惩罚工具过多的配置。tanh 把输入压到 -1 和 1 之间，参数越大提升越接近上限，这正是边际递减的意思。这个函数是人造的，只用来观察搜索机制；真实系统的评分来自实际运行。

先手算两个点，看看分数大致落在什么范围。$p=(0,1,2)$：$tanh(0)=0$，$tanh(0.25)≈0.245$，得分 $≈0.20+0.0245≈0.22$。$p=(6,8,3)$：$tanh(2)≈0.964$，$tanh(2/3)≈0.583$，得分 $≈0.20+0.30×0.964+0.10×0.964+0.05×0.583≈0.61$。让代码验证这两点，再做爬山与随机搜索两种方法。

In [ ]:
def agent_score(p):
    """三个参数 (推理步数, 集成样本数, 工具数) 的确定性评分。"""
    n_cot, n_samples, n_tools = p
    acc = 0.20 + 0.30 * np.tanh(n_cot / 3.0)
    acc += 0.10 * np.tanh(n_samples / 4.0)
    acc += 0.05 * np.tanh((n_tools - 2) / 1.5)
    acc -= 0.03 * max(0, n_tools - 5)
    return float(np.clip(acc, 0.0, 1.0))

for p in [(0, 1, 2), (6, 8, 3)]:
    print(f"agent_score{p} = {agent_score(p):.4f}")

def neighbors(p):
    """返回四个相邻参数点：推理步数与集成样本数各加减一。"""
    n_cot, n_samples, n_tools = p
    return [(n_cot + 1, n_samples, n_tools),
            (n_cot, n_samples + 1, n_tools),
            (max(0, n_cot - 1), n_samples, n_tools),
            (n_cot, max(1, n_samples - 1), n_tools)]

In [ ]:
def hill_climb(start, steps=12):
    """从 start 出发，每次移动到评分更高的最好邻居，返回轨迹。"""
    p = start
    trace = [(p, agent_score(p))]
    for _ in range(steps):
        best_c = max(neighbors(p), key=agent_score)
        if agent_score(best_c) <= agent_score(p):
            break
        p = best_c
        trace.append((p, agent_score(p)))
    return trace

def random_search(budget=30, seed=0):
    """独立采样 budget 个参数点，记录迄今最好评分。"""
    rng = np.random.RandomState(seed)
    best = -1.0
    trace = []
    for _ in range(budget):
        p = (int(rng.randint(0, 10)), int(rng.randint(1, 12)),
             int(rng.randint(0, 8)))
        s = agent_score(p)
        if s > best:
            best, best_p = s, p
        trace.append(best)
    return best_p, trace

hc_trace = hill_climb((0, 1, 2))
best_rs, rs_trace = random_search(30, seed=2)

print("爬山轨迹（参数, 评分）:")
for p, s in hc_trace:
    print(f"  {p}  {s:.3f}")
print("爬山终点评分:", round(hc_trace[-1][1], 3))
print("随机搜索最优参数:", best_rs, "评分:", round(agent_score(best_rs), 3))

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

_, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].plot([s for _, s in hc_trace], "o-", color="#2c7fb8")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Score")
axes[0].set_title("Hill climbing")
axes[1].plot(rs_trace, ".-", color="#d95f0e")
axes[1].set_xlabel("Sample")
axes[1].set_ylabel("Best score so far")
axes[1].set_title("Random search")
plt.tight_layout()
plt.show()
print("爬山从", round(hc_trace[0][1], 3), "爬到", round(hc_trace[-1][1], 3))

两条曲线讲的是两种搜索策略的差异。爬山从 (0, 1, 2) 出发，每步看四个邻居里谁评分最高，就移过去，最后停在 (7, 6, 2)，评分 0.585。随机搜索独立采样 30 个参数点，记录迄今最好的评分，最好的一次达到 (6, 10, 5)，评分 0.636。

两个结果对比着看。爬山每步都向上，路线连续，终点 0.585 比起点 0.224 好得多；但它只看得到紧挨着的四个邻居，视野局限于当前位置附近。评分函数在参数空间里可能有不止一个高点，爬山的终点未必是最高处。随机搜索不沿邻居走，而是到处撒点，样本够多时更可能落进评分更高的区域，本例里 0.636 就高过了爬山终点。

这个对比说明一件事：只靠局部修正，或者盲目乱试，都是粗糙的搜索。要让 Agent 的设计越过人工能想到的方案，需要把变异、选择、记忆组合成一个循环。下面进入 ADAS 的最小闭环。

前面两个小节都在参数空间里演示变异和选择，但 ADAS 真正的搜索空间是代码。这一小节回答一个问题：把变异、选择、记忆组合成一个闭环，循环该怎么转。最小闭环是四步：元 Agent 读 archive 摘要，写出一份 forward 函数代码；我们把它编译成可调用对象；在算术任务上评测得到准确率；把结果连同代码存进 archive。archive 从两份基线 Agent 起步，逐轮生长。先搭基础设施。

In [ ]:
import sys, os
_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, 'llm_client.py')):
    _root = os.path.dirname(_root)
    if _root == os.path.dirname(_root):
        break
if _root not in sys.path:
    sys.path.insert(0, _root)
from llm_client import get_llm

client = get_llm()
print("真实 API 演示:", False)

In [ ]:
def wrap_recipe(recipe):
    """把推理配方包成一份 forward 源码字符串。"""
    return f"""def forward(task):
    a, b = task
    return solve(a, b, recipe='{recipe}')"""

def compile_agent(src):
    """把 forward 源码编译成可调用函数，框架函数对生成的代码可见。"""
    ns = globals().copy()
    exec(src, ns)
    return ns["forward"]

def eval_agent(fn, problems=PROBLEMS, true=TRUE):
    """运行 Agent 在任务上评测，返回准确率。"""
    correct = sum(1 for (a, b), t in zip(problems, true)
                  if fn((a, b)) == str(t))
    return correct / len(problems)

# archive：字典，存"名字 → (源码, 得分)"。从两份基线起步，先算出真实得分。
archive = {"baseline_direct": (wrap_recipe("direct"), 0.0),
           "baseline_decompose": (wrap_recipe("decompose"), 0.0)}
for name, (src, _) in archive.items():
    archive[name] = (src, eval_agent(compile_agent(src)))
for name, (_, acc) in archive.items():
    print(f"{name}: 准确率 {acc:.2f}")

In [ ]:
def extract_recipe(src):
    """从源码读出配方名，解析失败返回 'unknown'。"""
    for rec in RECIPE_RATE:
        if f"recipe='{rec}'" in src:
            return rec
    return "unknown"

def meta_prompt(archive):
    """构造元 Agent 的提示词：读 archive 摘要，写新的 forward 代码。"""
    lines = []
    for name, (src, acc) in archive.items():
        lines.append("- " + name + ": 准确率 " + f"{acc:.2f}"
                     + ", 配方 " + extract_recipe(src))
    summary = "\n".join(lines)
    return ("你是元 Agent，负责设计能解两位数乘法的 Agent。框架提供 "
            "solve(a, b, recipe)，recipe 可取 direct / decompose / "
            "decompose_check / ensemble。已有 Agent：\n" + summary +
            "\n请写一个新的 forward(task) 函数，输出用 ```python 包裹。")

def generate_agent(client, archive, it):
    """元 Agent 生成一份 Agent 代码。真实 API 演示返回脚本化占位。"""
    if False:
        # 真实 API 演示输出为占位：从脚本化配方池轮转，展示步进石行为
        pool = ["decompose_check", "ensemble", "decompose", "decompose_check"]
        return wrap_recipe(pool[it % len(pool)])
    reply = client.chat([{"role": "user", "content": meta_prompt(archive)}])
    code = reply.split("```python")[-1].split("```")[0].strip()
    if "def forward" not in code:
        code = wrap_recipe("decompose_check")
    return code

print(meta_prompt(archive))
print("---- 元 Agent 输出（真实 API 演示为脚本化占位）----")
print(generate_agent(client, archive, 0))

In [ ]:
def adas_loop(client, archive, rounds=4):
    """元 Agent 搜索：生成 → 编译 → 评测 → 入库。返回每轮记录。"""
    log = []
    for it in range(rounds):
        src = generate_agent(client, archive, it)
        try:
            acc = eval_agent(compile_agent(src))
        except Exception:
            acc = 0.0
        name = "discovered_" + str(it)
        best = max(s for _, s in archive.values())
        added = acc > best
        if added:
            archive[name] = (src, acc)
        log.append((it, extract_recipe(src), acc, added))
    return log

log = adas_loop(client, archive, rounds=4)
for it, recipe, acc, added in log:
    flag = "入库" if added else "舍弃"
    print(f"第 {it} 轮: 配方 {recipe:<14} 准确率 {acc:.2f}  {flag}")

print("最终 archive（步进石记录）:")
for name, (src, acc) in archive.items():
    print(f"  {name:<18} 配方 {extract_recipe(src):<14} 准确率 {acc:.2f}")

把刚才的输出逐轮回放，看四步闭环每一步的输入和输出。

起点是 archive 里的两条基线：baseline_direct 准确率 0.55，baseline_decompose 准确率 0.80。这是记忆的初始内容，也是元 Agent 第一轮能读到的全部摘要。

第 0 轮的四步：
1. 变异。元 Agent 读 archive 摘要（两行：direct 0.55、decompose 0.80），写出新 forward：`return solve(a, b, recipe='decompose_check')`。
2. 编译。exec 把这段源码变成可调用的 forward。
3. 评测。在 40 道题上运行，准确率 0.93。
4. 入库。当前历史最优 best = max(0.55, 0.80) = 0.80，0.93 大于 0.80，入库，命名为 discovered_0。

第 1 轮：archive 现在有三条摘要，元 Agent 能看到 discovered_0 的 0.93，写出 ensemble。评测 1.00，best 已升到 0.93，1.00 大于 0.93，入库，命名为 discovered_1。

第 2 轮：元 Agent 写了 decompose。0.80 不超过当前 best 1.00，舍弃，archive 不变。

第 3 轮：写了 decompose_check。0.93 同样不超过 1.00，舍弃。

入库条件只有一条：`acc > 当前历史最优`。这保证了 archive 只记录真正的进步，重复或倒退的候选直接丢掉。最终 archive 稳定在四个条目：没有新的突破时，archive 就不再变化。

闭环的关键在最后一步入库。被存进 archive 的代码会出现在下一轮的提示词摘要里，直接影响元 Agent 下一次写什么。这就是下一轮基于评分改进的机制：评分高的代码留下成为新起点，评分低的代码消失，元 Agent 能组合的历史越来越长。

archive 的价值在于给变异算子当记忆。上面第 0 轮发现的 decompose_check 借用了基线里的 decompose，再补上验算；第 1 轮的 ensemble 又把 decompose_check 组合成投票。每次入库都建立在上一轮的基础上，后发现的 Agent 组合了前面 Agent 的组件，而不是从零开始。论文里把这种逐级累积的现象叫做步进石（stepping stone）。本节的 archive 从两个基线出发，收录两个改进后停止增长：没有新突破时，archive 不再变化。

步进石这个名字来自过河的比喻：踩着河里的石头一步一步过河，每一块石头都是下一步的立足点。进化里的步进石是同一个意思，一个解成为构造下一个解的原料。

用本节的 archive 看这条链：

```text
baseline_decompose (0.80)
        ↓ 借用 decompose 的思路
discovered_0: decompose_check (0.93)   # decompose 补上验算
        ↓ 把 decompose_check 包成投票
discovered_1: ensemble (1.00)          # 多个配方同时跑再投票
```

discovered_0 不是凭空设计出来的，它是在 baseline_decompose 的基础上补上了验算；discovered_1 又把 discovered_0 包装成 ensemble。每一步都复用了前一步的成果，这正是 archive 当记忆的功劳。没有 archive，每一轮元 Agent 只能从零设计；有 archive，它能读到历史最优的代码，沿着已有台阶继续往上踩。

论文里 ARC 挑战的数据同样如此：第 3 轮出现多路 CoT 加修正，第 25 轮才把多种反馈组装成最终 Agent。早期发现的多是简单组件，越到后面越能组合出复杂方案。步进石机制决定了开放进化不是每次从头搜一遍，而是沿着历史累积的台阶持续前进，这也是它被称为开放进化的原因。

这一节回答一个问题：把第一节的循环从设计 Agent 放大到做科研，循环该怎么转。ADAS 把设计 Agent 放进了循环。AI Scientist 把同样的循环放大到完整的科研流程：提出研究 idea，写代码做实验，把结果写成论文，由自动评审器打分，再把通过的 idea 连同分数存进知识 archive。与 ADAS 相比有三个升级：变体从 Agent 代码变成研究 idea，适应度从任务准确率变成论文评审，种群从代码库变成知识库。

自动评审器是闭环的关键，也是风险所在。论文里用按 NeurIPS 指南打分的 GPT-4o 评审 agent，在 500 篇 ICLR 2022 论文上达到接近人类水平的判断。当作者与评审都是同一个 AI 时，闭环里没有外部真值，也就是没有一个系统之外的客观标准来裁定结果对不对。我们用最小循环观察这个结构：大模型提出 idea，确定性环境跑出实验指标，评审函数决定取舍。

In [ ]:
# 实验环境：固定回归数据上从零实现的全批量梯度下降。
# idea 通过修改超参改变实验指标（验证损失），全部确定可复现。
import numpy as np
np.random.seed(42)

rng = np.random.RandomState(0)
X = rng.randn(80, 8)
w_true = np.array([1.0, 2.0, -1.5, 1.0, -0.5, 0.8, -0.3, 0.6])
y = X @ w_true + 0.6 * rng.randn(80)
Xtr, Xva = X[:60], X[60:]
ytr, yva = y[:60], y[60:]

def fit_linear(Xtr, ytr, Xva, yva, lr=0.05, wd=0.0, epochs=5, momentum=False, optimizer=None, learning_rate=None, **kwargs):
    """全批量梯度下降拟合线性模型，返回验证集均方误差。"""
    if learning_rate is not None:
        lr = learning_rate
    w = np.zeros(8)
    v = np.zeros(8)
    n = len(ytr)
    for _ in range(epochs):
        g = 2.0 * (Xtr.T @ (Xtr @ w - ytr)) / n + 2.0 * wd * w
        v = 0.9 * v + g
        w = w - lr * (v if momentum else g)
    pred = Xva @ w
    return float(np.mean((pred - yva) ** 2))

# 基线只训练 5 轮，明显欠训练，给 idea 留下提升空间
BASE_CFG = {"lr": 0.05, "wd": 0.0, "epochs": 5, "momentum": False}
baseline_val = fit_linear(Xtr, ytr, Xva, yva, **BASE_CFG)
print("基线配置验证损失:", round(baseline_val, 4))

IDEAS = [("降低学习率到 0.005", {"lr": 0.005}),
         ("加入 L2 权重衰减", {"wd": 0.05}),
         ("采用动量优化", {"momentum": True}),
         ("训练轮数翻倍", {"epochs": 10}),
         ("训练轮数四倍", {"epochs": 20})]
for desc, cfg in IDEAS:
    val = fit_linear(Xtr, ytr, Xva, yva, **dict(BASE_CFG, **cfg))
    print(f"{desc}: 验证损失 {val:.4f}")

In [ ]:
# idea 生成与评审都走大模型：真实 API 演示返回脚本化占位。
def generate_idea(client, it):
    """提出一个研究 idea，返回 (描述, 超参改动)。"""
    if False:
        # 真实 API 演示输出为占位：从候选池轮转
        return IDEAS[it % len(IDEAS)]
    prompt = ("你是研究助手。基线用 5 轮全批量梯度下降拟合线性回归，"
              "验证损失 " + f"{baseline_val:.4f}" + "。提出一个改进"
              "超参配置的 idea，按两行输出：\nidea: <一句话>\ncfg: <python dict>")
    reply = client.chat([{"role": "user", "content": prompt}])
    return parse_idea(reply) or IDEAS[it % len(IDEAS)]

def parse_idea(reply):
    """从回复里解析 (描述, 超参 dict)，解析失败返回 None。"""
    desc, cfg = None, None
    for ln in reply.splitlines():
        if ln.startswith("idea:") or ln.startswith("idea："):
            desc = ln.split(":", 1)[-1].strip()
        if ln.startswith("cfg:") or ln.startswith("cfg："):
            cfg = ln.split(":", 1)[-1].strip()
    if desc is None or cfg is None:
        return None
    try:
        cfg = eval(cfg)
    except Exception:
        return None
    return (desc, cfg) if isinstance(cfg, dict) else None

def parse_scores(reply):
    """从评审回复里解析分数 dict，解析失败返回 None。"""
    out = {}
    for token in reply.replace(",", " ").split():
        if ":" in token:
            k, v = token.split(":", 1)
        elif "=" in token:
            k, v = token.split("=", 1)
        else:
            continue
        if k in ("soundness", "presentation", "contribution", "overall"):
            try:
                out[k] = float(v)
            except ValueError:
                pass
    return out if len(out) == 4 else None

def review_idea(client, desc, val, seen, baseline):
    """评审一个 idea，返回 (分数 dict, 是否接受)。

    真实 API 演示按规则打分：损失改善越多分数越高，重复 idea 直接拒绝。
    """
    improve = baseline - val
    if True:
        prompt = ("评审一个研究 idea。描述：" + desc + "\n验证损失："
                  + f"{val:.4f}，基线：" + f"{baseline:.4f}。\n"
                  "请按 NeurIPS 指南给出分数：soundness, presentation, "
                  "contribution, overall（1-10）。")
        reply = client.chat([{"role": "user", "content": prompt}])
        parsed = parse_scores(reply)
        if parsed:
            parsed["accept"] = (desc not in seen and parsed["overall"] >= 6
                                and parsed["contribution"] >= 3)
            return parsed
    soundness = float(np.clip(3 + 50 * improve, 1, 10))
    contribution = float(np.clip(1 + 40 * improve, 1, 10))
    novelty = 0.3 if desc in seen else 0.9
    overall = float(np.clip(0.5 * soundness + 0.3 * contribution
                            + 0.2 * novelty, 1, 10))
    accept = desc not in seen and overall >= 6 and contribution >= 3
    return {"soundness": round(soundness, 1), "presentation": 5.0,
            "contribution": round(contribution, 1), "overall": round(overall, 1),
            "accept": accept}

demo = review_idea(client, "采用动量优化", baseline_val / 3, set(), baseline_val)
print("评审示例:", demo)

评审器的打分来自一个确定性公式，方便手算。先算改进量 improve = 基线验证损失减去本 idea 的验证损失，improve 越大，说明这个 idea 让结果变好得越多。再把 improve 映射到三个分数维度：

```text
soundness    = clip(3 + 50×improve, 1, 10)     # 结果有多可靠
contribution = clip(1 + 40×improve, 1, 10)     # 贡献有多大
novelty      = 0.9（首次出现的 idea）或 0.3（重复的 idea）
overall      = clip(0.5×soundness + 0.3×contribution + 0.2×novelty, 1, 10)
```

验证损失降得越多，improve 越大，soundness 和 contribution 越高；idea 让损失变差时 improve 为负，两个分数都被 clip 到下限 1。clip 的意思是数值超出区间时被压到边界上，这里区间是 1 到 10。novelty 是独立维度，重复出现过的 idea 直接被压到 0.3。

用手算验证评审示例。demo 里 val = 基线 ÷ 3 = 2.1295 ÷ 3 ≈ 0.710，improve = 2.1295 − 0.710 ≈ 1.420。代入公式：

```text
soundness    = clip(3 + 50×1.420, 1, 10) = clip(74, 1, 10) = 10
contribution = clip(1 + 40×1.420, 1, 10) = clip(58, 1, 10) = 10
novelty      = 0.9
overall      = clip(0.5×10 + 0.3×10 + 0.2×0.9, 1, 10) = 8.18 ≈ 8.2
```

与输出 {soundness: 10.0, contribution: 10.0, overall: 8.2, accept: True} 一致。accept 的门槛是 overall ≥ 6 且 contribution ≥ 3：overall 反映总体质量，contribution 单独把关，防止写得漂亮但没有贡献的 idea 通过。

三个维度对应论文评审的三个问题：结论可不可信、对领域有没有新东西、是不是重复劳动。真实 AI Scientist 里评审读的是 AI 写出的整篇论文，这里压缩成一组数字，机制是相同的。

In [ ]:
def scientist_loop(client, rounds=7):
    """idea → 实验 → 评审 → 入库，返回每轮记录与知识 archive。"""
    archive = {}
    seen = set()
    log = []
    if False:
        print("idea 由 脚本化示例 大模型脚本化生成，评审按规则打分（占位输出）")
    for it in range(rounds):
        desc, cfg = generate_idea(client, it)
        val = fit_linear(Xtr, ytr, Xva, yva, **dict(BASE_CFG, **cfg))
        review = review_idea(client, desc, val, seen, baseline_val)
        dup = desc in seen
        if review["accept"]:
            archive[desc] = {"cfg": cfg, "val": val, "review": review}
        seen.add(desc)
        log.append((it, desc, val, review["overall"], review["accept"], dup))
    return log, archive

log, archive = scientist_loop(client, rounds=7)
for it, desc, val, overall, acc, dup in log:
    if dup:
        reason = "舍弃（重复 idea）"
    else:
        reason = "入库" if acc else "舍弃（分数不足）"
    print(f"第 {it} 轮  {desc:<10} val={val:.4f} overall={overall}  {reason}")
print("知识 archive（通过的 idea）:")
for desc, info in archive.items():
    print(f"  {desc}: val={info['val']:.4f} overall={info['review']['overall']}")

回放 7 轮，看 idea、实验、评审、入库这条闭环。基线的验证损失是 2.1295。每个 idea 先改超参，再跑实验得到新的验证损失，最后交给评审。

前两轮是反面例子。第 0 轮的 idea 把学习率降到 0.005，验证损失反而涨到 4.7076。improve 为负，soundness 被 clip 到 1，overall 只有 1.0，达不到 6，舍弃。第 1 轮加入 L2 权重衰减，损失 2.1516，比基线还略差，overall 1.4，同样舍弃。验证损失变差的 idea 进不了 archive。

第 2 轮采用动量优化，把损失降到 0.5683，improve ≈ 1.56，soundness 和 contribution 都被 clip 到 10，overall 8.2，入库。第 3、4 轮训练轮数翻倍、四倍，分别把损失降到 1.0247 和 0.4568，都达到门槛，入库。知识 archive 最终收录这三个 idea。

第 5、6 轮值得注意。元 Agent 又提出降低学习率和 L2 权重衰减，这两个 idea 已在 seen 里记录过，novelty 降到 0.3，评审判为重复，overall 分别只有 1.0 和 1.3，没有通过。这是去重机制：同一个 idea 只评审一次，防止 archive 里塞满重复条目。

对比 ADAS 的闭环，结构是相同的，只是三处设置的取值不同。ADAS 的变异结果是代码，适应度是任务准确率，种群是代码 archive；AI Scientist 的变异结果是研究 idea，适应度是论文评审分数，种群是知识 archive。

风险也在这里。评审者与作者都是 AI，闭环里没有外部真值，评审的 8.2 分是否真的对应科学价值，没有外部对象来裁定。论文里就出现过 AI 把负结果写成改善的幻觉。这个结构性问题放到第 4 节讨论。

这一节回答一个问题：把一份完整的算法源码当作可以进化的对象，怎么从它出发搜出更好的代码。AI Scientist 的变体是研究 idea，AlphaEvolve 又把变体换回代码，但这次不是第一节里一份单薄的 Agent 函数，而是整份算法源码。大模型是变异算子，它读当前程序，输出一段代码改动，这种改动的格式叫 diff；自动评测器计算适应度；进化数据库按"行为尽量多样"的思路维持一批不同解的种群。三个组件与 ADAS 相同，区别在规模：可以进化整份文件、任意语言、小时级的并行评测。

论文里最有代表性的结果是矩阵乘法：用 48 次标量乘法算出两个 4×4 复矩阵的积，这是自 1969 年 Strassen 的 49 次以来 56 年的首个改进，整个发现只用了约 15 次变异。进化还有个硬前提：适应度必须能被自动评测。AlphaEvolve 的每个解都要在评测器里跑若干小时，无法自动算的问题就进不了循环，这既是它的边界，也是它从 FunSearch 只进化单个函数扩到整份文件的原因。先看变异算子的载体，一段 diff 长什么样。

AlphaEvolve 复用 ADAS 的三件套，只是把变体从一份 Agent 代码换成整份算法源码。这里换上遗传学的词汇，从最基本的说起。遗传学里，生物的全部性状都由一段遗传物质决定，这段物质叫基因组；生物长成什么样、怎么行动，是这份基因组运行出来的结果，叫表现型。放到代码上：一段可执行的源码就是基因组，程序跑起来的行为就是表现型，评测器算出的适应度决定它能不能继续繁殖。

基因组、变异、选择、种群四件套在代码空间里分别对应：

- 基因组：一段可执行的源码。AlphaEvolve 进化的可以是整份文件、任意语言。
- 变异：大模型读当前程序，输出一段代码改动，用 SEARCH/REPLACE 的 diff 格式表达，先写要替换的原代码，再写替换成什么。
- 适应度：自动评测器跑出来的分数。
- 种群：进化数据库，用质量多样性的思路维持一批行为不同的解，而不是只留一个最优。

变异选 diff 而不是整份重写，因为 diff 是局部改动。一段程序的大部分代码是基础设施，只有一小块决定行为；只动这一小块，程序其余部分保持稳定，变异成功的概率更高。反过来，整份重写容易把之前积累的好结构全打散。把代码当遗传物质来理解：变异只改基因的一小段，遗传物质整体延续，这正是生物进化的方式。

下面用一段冒泡排序代码演示 diff 变异。

In [ ]:
# 变异算子：解析并应用 AlphaEvolve 的 SEARCH/REPLACE diff 块。
def apply_diff(old_code, search, replace):
    """在 old_code 中把 search 文本替换成 replace，返回新代码。"""
    if search not in old_code:
        raise ValueError("SEARCH 块未在代码中找到")
    return old_code.replace(search, replace)

SRC = """def bubble(nums):
    for i in range(len(nums) - 1):
        for j in range(len(nums) - 1 - i):
            if nums[j] > nums[j + 1]:
                nums[j], nums[j + 1] = nums[j + 1], nums[j]
    return nums
"""

def eval_sort(src):
    """运行排序代码，返回在 3 组数组上的正确比例。"""
    ns = {}
    exec(src, ns)
    fn = ns["bubble"]
    cases = [[3, 1, 2], [5, 4, 3, 2, 1], [1, 2, 3]]
    ok = 0
    for c in cases:
        ok += int(fn(list(c)) == sorted(c))
    return ok / len(cases)

print("原始代码正确率:", eval_sort(SRC))

# 变异 1：把升序比较改成降序比较，排序方向反转
search = "if nums[j] > nums[j + 1]:"
replace = "if nums[j] < nums[j + 1]:"
mut1 = apply_diff(SRC, search, replace)
print("变异 diff：")
print(f"<<<<<<< SEARCH\n{search}\n=======\n{replace}\n>>>>>>> REPLACE")
print("变异后正确率:", eval_sort(mut1))

# 变异 2：外层循环多扫一轮，行为不变
mut2 = apply_diff(SRC, "range(len(nums) - 1)", "range(len(nums))")
print("去掉外层 -1 后正确率:", eval_sort(mut2))

两个变异的对比展示了 diff 变异的两面。原始冒泡排序三组用例全对，正确率 1.0。

变异 1 把比较符从 `>` 改成 `<`，排序方向从升序变成降序。手算第一组 [3, 1, 2]：排序把较小的数往后交换，得到 [3, 2, 1]，而正确结果是 [1, 2, 3]，错；三组用例全部反转，正确率 0.0。一个符号的改动足以让适应度崩塌。代码变异不像参数变异那样平滑，改错位置就全盘出错。

变异 2 把外层循环 `range(len(nums) - 1)` 改成 `range(len(nums))`，多扫一轮。最后一轮内层循环范围是空，什么都不做，结果不变，正确率仍是 1.0。有些改动不改变结果、适应度也不变，这种变异叫中性变异。它让种群多了一个行为相同的变体，本身不改变分数，却可能成为下一步组合的素材。

变异算子的输出就是 SEARCH/REPLACE 这种格式：SEARCH 块是要找的原代码，REPLACE 块是替换成的新代码，替换发生在匹配到的位置。真实 AlphaEvolve 里大模型生成的就是这种 diff，评测器只跑替换后的新程序。

这一小节回答一个问题：一批个体放在一起进化，选择怎么让种群整体变好。diff 展示了单次变异，进化还需要种群与选择：一批个体同时进化，每代里评分高的个体留下并复制，评分低的被淘汰，这套流程就是遗传算法。下面在可解析验证的小目标上跑进化，用一组傅里叶基系数去逼近目标函数 sin x + 0.5 cos 2x。傅里叶基是一组固定的函数，把这些函数的加权和算出来去贴合目标函数，权重就是每个个体要进化的一组系数。每个个体是一段 6 维系数向量，变异是在系数上加一点随机扰动，选择是保留评分最高的两个个体作为精英并复制。评分用负均方误差，越大越好。

In [ ]:
# 目标：用傅里叶基系数逼近 sin x + 0.5 cos 2x。
# 个体是 6 个系数，评分是负均方误差，越大越好。
x = np.linspace(-np.pi, np.pi, 200)
y_target = np.sin(x) + 0.5 * np.cos(2 * x)

def basis_at(x):
    """傅里叶基：1, x, sin x, cos x, sin 2x, cos 2x。"""
    return np.stack([np.ones_like(x), x, np.sin(x), np.cos(x),
                     np.sin(2 * x), np.cos(2 * x)])

B = basis_at(x)

def fitness(coefs):
    """系数向量在网格上的逼近质量：负均方误差。"""
    pred = B.T @ coefs
    return -float(np.mean((pred - y_target) ** 2))

coef_star, *_ = np.linalg.lstsq(B.T, y_target, rcond=None)
print("最小二乘最优评分的对照值:", round(fitness(coef_star), 4))

def evolve(pop, gen, elite=2, sigma=0.3, seed=0):
    """进化 gen 代：每代选最优 elite 个，用变异复制填满种群。"""
    rng = np.random.RandomState(seed)
    curve = []
    for _ in range(gen):
        scores = np.array([fitness(p) for p in pop])
        curve.append(float(scores.max()))
        order = np.argsort(scores)[::-1][:elite]
        parents = [pop[i] for i in order]
        children = []
        while len(children) < len(pop) - elite:
            parent = parents[rng.randint(elite)]
            children.append(parent + sigma * rng.randn(len(parent)))
        pop = parents + children
    return curve, pop

rng0 = np.random.RandomState(1)
pop = [rng0.randn(6) * 2 for _ in range(30)]
curve, last_pop = evolve(pop, gen=60, elite=2, sigma=0.3, seed=2)
print("第 0 代最优评分:", round(curve[0], 4))
print("第 60 代最优评分:", round(curve[-1], 4))

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 3.5))
plt.plot(curve, color="#2c7fb8")
plt.xlabel("Generation")
plt.ylabel("Best fitness")
plt.title("Best fitness over generations")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# 质量多样性视图：把最后种群按两个行为特征分箱，每箱保留最高评分
b0 = np.array([p[0] for p in last_pop])
b1 = np.array([p[1] for p in last_pop])
fs = np.array([fitness(p) for p in last_pop])
grid = np.full((20, 20), np.nan)
ix = np.clip(((b0 + 2.5) / 5.0 * 19).astype(int), 0, 19)
iy = np.clip(((b1 + 2.5) / 5.0 * 19).astype(int), 0, 19)
for a, b, f in zip(ix, iy, fs):
    grid[a, b] = f if np.isnan(grid[a, b]) else max(grid[a, b], f)

plt.figure(figsize=(5.5, 4.5))
im = plt.imshow(grid, origin="lower", cmap="viridis")
plt.colorbar(im, label="Fitness")
plt.xlabel("Behavior: coef[0]")
plt.ylabel("Behavior: coef[1]")
plt.title("Performance map of final population")
plt.tight_layout()
plt.show()
print("被占用的行为格数:", int(np.sum(~np.isnan(grid))), "/ 400")

先看左图，最优适应度随代数上升。第 0 代种群是 30 个随机系数向量，最好只有 -2.3756；到第 60 代达到 -0.0072，接近最小二乘的理论最优 -0.0。能逼近到这个值，因为目标函数 sin x + 0.5 cos 2x 恰好能被傅里叶基组合出来，基里含 sin x 和 cos 2x，存在一组系数让误差为 0。进化不需要知道这一点，它只靠评分高的留下来，一步步逼近。

把一代的流程拆开走一遍。种群 30 个个体，每个是 6 维系数向量：

1. 计算适应度。对每个个体算负均方误差，得 30 个分数。
2. 选择。取分数最高的 2 个当精英，其余 28 个淘汰。
3. 变异复制。从 2 个精英里随机抽 1 个当父本，加 σ = 0.3 的高斯噪声，生成 28 个子代。
4. 填满。精英 2 个加子代 28 个等于 30 个，进入下一代。

子代数 = 种群数 − 精英数 = 30 − 2 = 28。保留精英的同时还要加噪声，因为两者分工不同。保留精英是利用，分数最高的解直接继承，不倒退；加噪声是探索，在最优解附近扰动，寻找可能更好的邻居。两者缺一，要么原地不动，要么退化成随机搜索。σ 控制探索半径，σ 越大步子越猛，σ 越小越精细，0.3 在本例里平衡了两者。

右图回答另一个问题：最后一代的 30 个个体挤在哪里。把每个个体的前两个系数当行为特征，落入 20×20 的行为网格，每格只留最高分。结果只有 16 格被占用，种群没有挤成一团，而是散布在行为空间的不同区域。一个解可能在自己附近已经找不到更好的，但别处还有更高分的解，这种解叫局部最优。只保留一个最高分解，种群容易全都停在同一个局部最优。质量多样性是不只看谁分高，还看谁行为不同，保留一批行为不同的解，AlphaEvolve 用同样的思想维持进化数据库。

循环本身能工作，风险集中在评测函数上。这一节回答一个问题：当进化自己评价自己的产出时，会出什么错，怎么挡住。变异算子只优化它拿到的分数，一旦分数与真实目标脱节，进化就会找到取巧的路径，得分虚高却没有真正完成任务，这种行为有一个名字，叫奖励黑客。我们用一个演示暴露评测漏洞：Agent 直接记住评测集的答案，就能拿到满分。再把评测集与开发集分开，看这道护栏如何挡住取巧路径。

奖励黑客说的是：进化只优化它看到的分数，不优化分数背后的真实目标。一旦分数能被取巧刷高，而取巧并没有真的完成任务，进化就会走向取巧，因为取巧的分数更高。

用一个具体例子把机制讲透。假设用来挑方案的 40 道题是固定的，从 (11, 11) 到 (15, 18)，这批题叫开发集。诚实 Agent 用 decompose_check 配方算每一题，评测得 0.93。现在换一种 Agent，它不计算，而是把 40 道题的答案抄下来存进一张表，评测时按题目查表返回。开发集 40 题全部命中，得 1.00，比诚实 Agent 还高。

只看开发集分数，进化会留下抄答案的 Agent，因为 1.00 大于 0.93。但换一批没见过的 40 道题，从 (20, 21) 到 (24, 28)，抄答案的 Agent 查不到任何一条，只能返回占位答案，全错，得 0.00；诚实 Agent 还是 0.93，因为它学的不是答案表，而是乘法算法。

手算核对。开发集 40 题，抄答案全对，40 / 40 = 1.00；独立新题 40 题查不到答案，假设返回 "0"，没有一道正确，0 / 40 = 0.00。同一个 Agent，两个分数差距巨大。问题不在 Agent 聪明不聪明，而在分数从哪里来。开发题与评测题重合时，记住答案就是最优策略。

In [ ]:
# 奖励黑客演示：开发与评测共用同一份题目时，硬编码答案是捷径。
import numpy as np

A = np.repeat(np.arange(11, 16), 8)
B = np.tile(np.arange(11, 19), 5)
TRAIN = list(zip(A.tolist(), B.tolist()))
TRUE_TRAIN = [a * b for a, b in TRAIN]

H = np.repeat(np.arange(20, 25), 8)
K = np.tile(np.arange(21, 29), 5)
HOLD = list(zip(H.tolist(), K.tolist()))
TRUE_HOLD = [a * b for a, b in HOLD]

def honest_agent(task):
    """诚实 Agent：用框架的分步配方求解。"""
    a, b = task
    return solve(a, b, "decompose_check")

answer_table = {(a, b): str(a * b) for (a, b) in TRAIN}

def cheat_agent(task):
    """取巧 Agent：把开发题答案硬编码，新题一律答 0。"""
    return answer_table.get(task, "0")

print("诚实 Agent  开发题准确率:  ", round(eval_agent(honest_agent, TRAIN, TRUE_TRAIN), 2))
print("诚实 Agent  独立新题准确率:", round(eval_agent(honest_agent, HOLD, TRUE_HOLD), 2))
print("取巧 Agent  开发题准确率:  ", round(eval_agent(cheat_agent, TRAIN, TRUE_TRAIN), 2))
print("取巧 Agent  独立新题准确率:", round(eval_agent(cheat_agent, HOLD, TRUE_HOLD), 2))

输出把问题摆得很清楚。诚实 Agent 在开发集和独立新题上都是 0.93，它掌握的算法不依赖具体题目，换一批题照样对。取巧 Agent 在开发集上 1.0，是四行里最高的分数，但在独立新题上 0.0，是四行里最低的。

如果这一节只评测开发集，进化会留下取巧 Agent，因为它分数最高。可它根本没学会乘法，只是背下了一份答案，真实目标（解任意两位数的乘法）一个都没达成。这就是奖励黑客：分数被优化到最高，真实目标没有进步。

这里要澄清一个常见误解。取巧 Agent 不是在作弊，它只是做完了被要求的事，在给定评测上拿高分。进化没有意图，它只放大得分高的方案。分数定义错了，优化方向就错了，这不是 Agent 的错，是评测设计的错。修复的办法只有一个，让分数来自一个取巧够不到的地方。

In [ ]:
# 修复：开发集与评测集分离，评测集从不参与选择。
# 用开发集挑配方，再用独立评测集报告真实表现。
def run_search_with_holdout(dev, hold, rounds=4):
    """开发集上搜索，独立评测集上报。返回 (开发最优配方, 开发准确率, 独立准确率)。"""
    dev_problems, dev_true = dev
    hold_problems, hold_true = hold
    pool = ["decompose_check", "ensemble", "direct", "decompose_check"]
    best_recipe, best_acc = None, -1.0
    for it in range(rounds):
        recipe = pool[it % len(pool)]
        acc = eval_agent(compile_agent(wrap_recipe(recipe)),
                         dev_problems, dev_true)
        if acc > best_acc:
            best_recipe, best_acc = recipe, acc
    hold_acc = eval_agent(compile_agent(wrap_recipe(best_recipe)),
                          hold_problems, hold_true)
    return best_recipe, best_acc, hold_acc

recipe, dev_acc, hold_acc = run_search_with_holdout((TRAIN, TRUE_TRAIN),
                                                     (HOLD, TRUE_HOLD))
print("开发集上选出的配方:", recipe, " 开发集准确率:", round(dev_acc, 2))
print("独立评测集准确率:", round(hold_acc, 2))
print("结论：硬编码答案在独立评测上得 0 分，评测独立性挡住取巧路径。")

修复的做法是把开发集与评测集彻底分开。开发集参与选择，在它上面跑各个配方，挑准确率最高的；评测集从不参与选择，只用来报告最终选出的方案在没见过的题上的表现。硬编码答案的 Agent 在开发集上 1.0，会被选中，但轮到评测集报告时，它一条都查不到，得 0.00，被这道护栏挡在门外。

输出里，开发集选出了 ensemble 配方，开发集 1.0，独立评测集也是 1.0。ensemble 能通过，因为它不是记忆，它把多个配方同时跑再投票，开发集上没有的题照样算得对。开发集分数高、评测集分数也高，说明这个分数真的学到了东西。

这背后是机器学习里最古老的分割思想：训练集用来调参数，测试集用来报告真实表现。Agent 进化的开发集对应训练集，独立评测集对应测试集。三条原则可以记下来：评测集永远不参与选择；评测题与开发题分布相同但不重复；报告成绩只用评测集。守住这三条，奖励黑客就无处藏身。

风险不止奖励黑客。AI Scientist 的评审者与作者同为 AI，闭环缺少外部真值。论文里就出现过 AI 把负结果写成改善、声称使用了错误硬件等幻觉；当 AI 作者批量投稿时，审稿系统也会被压垮。

三篇论文各自给出工程护栏：ADAS 建议在容器里执行生成的代码，AI Scientist 建议沙箱化并标注 AI 产出，AlphaEvolve 承认评测必须可算本身就是前提。综合三者，结论是：开放进化能让 Agent 越过人设计的天花板，也可能走向不受控的方向，评测的独立性是唯一的护栏。

## 小结

这一节围绕"让 Agent 改进自己"展开。所学内容：

- [ ] 开放进化的三要素：变异算子、选择机制、种群与 archive
- [ ] 变异的最小演示：改提示词的一个词，评分就变
- [ ] 参数空间搜索：爬山与随机搜索在确定性评分函数上的轨迹
- [ ] ADAS 最小闭环：元 Agent 写 forward 代码 → 编译 → 评测 → 入库
- [ ] 步进石：后发现的 Agent 组合前面 Agent 的组件，archive 给变异算子当记忆
- [ ] AI Scientist 把闭环放大到科研：idea → 实验 → 评审，适应度是论文评审
- [ ] AlphaEvolve 以整份代码为基因组，diff 是变异算子，评测器是适应度
- [ ] 失败模式：奖励黑客来自评测漏洞，评测独立性是唯一的护栏

## 作业

> 可以让 AI 帮忙解释思路，但不建议直接让 AI "做完这道题"。

三道题各有一处空位，参考答案已填入代码，先在心里手算，再运行核对 assert。

**作业 1：补全适应度与选择**

给定 `scores = {"a": 0.3, "b": 0.9, "c": 0.6, "d": 0.4}`，补全 `select_top_k(scores, k=2)`，返回得分最高的 2 个 key，按得分从高到低；再补全 `archive_update(archive, name, score)`，只在 score 超过 archive 当前最低分时入库。

小提示：用 `sorted(scores.items(), key=lambda kv: kv[1], reverse=True)` 排序后切片；入库前与 `min(archive.values())` 比较。

In [ ]:
def select_top_k(scores, k=2):
    """返回得分最高的 k 个 key，按得分从高到低。"""
    ordered = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    return [name for name, _ in ordered[:k]]   # 空位在这里

def archive_update(archive, name, score):
    """score 超过 archive 当前最低分时入库，返回是否入库。"""
    if not archive:
        archive[name] = score
        return True
    if score > min(archive.values()):   # 空位在这里
        archive[name] = score
        return True
    return False

scores = {"a": 0.3, "b": 0.9, "c": 0.6, "d": 0.4}
assert select_top_k(scores, k=2) == ["b", "c"], "top-2 应是 b 与 c"

arch = {"x": 0.5}
assert archive_update(arch, "y", 0.9) is True
assert archive_update(arch, "z", 0.4) is False
assert "z" not in arch
print("收获：按得分选 top-k，只在超过当前最低分时入库。")

**作业 2：补全 Agent 代码的编译与调用**

元 Agent 返回的是一段字符串源码。补全 `compile_agent(src)`：用 `exec` 把源码编译进命名空间，取出 `forward` 并返回；再补全 `run_agent(fn, task)`，调用它返回答案。给定源码里先定义好 `forward(task)`。

小提示：`exec(src, ns)` 之后 `ns["forward"]` 就是可调用对象。

In [ ]:
def compile_agent(src):
    """把 forward 源码字符串编译成可调用函数。"""
    ns = {}
    exec(src, ns)
    return ns["forward"]   # 空位在这里

def run_agent(fn, task):
    """调用 Agent 处理一个任务，返回答案字符串。"""
    return fn(task)   # 空位在这里

src = """def forward(task):
    a, b = task
    return str(a * b)
"""
agent = compile_agent(src)
assert callable(agent), "编译结果应可调用"
assert run_agent(agent, (7, 8)) == "56"
print("收获：元 Agent 生成的源码可以 exec 成可调用函数并执行。")

**作业 3：补全评审打分的阈值决策**

评审返回分数 dict。补全 `should_accept(review, threshold=6)`：`overall` 达到阈值且 `contribution` 不低于 3 时接受，否则拒绝；再补全 `merge_reviews(reviews)`，对多个评审的 `overall` 取平均。

小提示：`overall` 与 `contribution` 直接取键；平均用 `sum(...) / len(...)`。

In [ ]:
def should_accept(review, threshold=6):
    """overall 达到阈值且 contribution 不低于 3 时接受。"""
    return review["overall"] >= threshold and review["contribution"] >= 3   # 空位在这里

def merge_reviews(reviews):
    """多个评审的 overall 取平均。"""
    return sum(r["overall"] for r in reviews) / len(reviews)   # 空位在这里

rev = {"soundness": 5, "presentation": 4, "contribution": 6, "overall": 6}
assert should_accept(rev) is True, "overall 6 且 contribution 6 应接受"
assert should_accept({"overall": 6, "contribution": 2}) is False
assert merge_reviews([{"overall": 5}, {"overall": 7}]) == 6.0
print("收获：评审阈值判断与多评审平均。")

## 参考资料

- Hu et al., [Automated Design of Agentic Systems](https://arxiv.org/abs/2408.08435), 2024 — 元 Agent 在代码空间里设计 Agent，本讲三要素的第一档。代码 https://github.com/ShengranHu/ADAS
- Lu et al., [The AI Scientist: Towards Fully Automated Open-Ended Scientific Discovery](https://arxiv.org/abs/2408.06292), 2024 — idea → 实验 → 论文 → 评审的全自动科研闭环。代码 https://github.com/SakanaAI/AI-Scientist
- Novikov et al., [AlphaEvolve: A coding agent for scientific and algorithmic discovery](https://arxiv.org/abs/2506.13131), 2025 — 以代码为基因组、以评测器为适应度的进化式编码 Agent
- Romera-Paredes et al., [FunSearch: Mathematical discoveries from program search with LLMs](https://arxiv.org/abs/2312.02174), 2023 — AlphaEvolve 的前身，LLM 进化单个函数
- Wang et al., [Quality-Diversity algorithms: A generic definition and an illustration](https://arxiv.org/abs/2103.04313), 2021 — MAP-Elites 与质量多样性思想，AlphaEvolve 种群的算法来源
- Clune, [AI-Generating Algorithms: An Alternate Paradigm](https://arxiv.org/abs/1901.01346), 2019 — 本讲理论源头，AI-GA 三支柱
- 本仓库 `llm_client.py` — 所有 LLM 演示的统一入口，真实 API 演示保证离线可执行